<a target="_blank" href="https://colab.research.google.com/github/AndreiSokolovskii/hackaton_december_2025/blob/main/Hackaton_december_2025.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Hackaton December 2025 for <font color="red">G</font>raph <font color="red">T</font>ransformer <font color="red">I</font>nverse <font color="red">F</font>olding *<font color="red">D</font>e <font color="red">N</font>ovo* <font color="red">P</font>rotein <font color="red">D</font>esign <font color="red">M</font>odel**

Or simply <font color="red">GTIFDNPDM</font>

**Introduction.**
*   The notebook is still in development.
*   At this moment, the following  prediction regimes are available:

--------------------------------------------

**3D Protein backbone structure ⇒ Sequence**
1. full sequence prediction from backbone blueprint
2. partiall sequence design from partially defined structure


**Important notice**
* WIP





In [ ]:
%%capture
#@title #Installation required libraries, downloading model weights.
#@markdown ESMfold could be used for designed sequnces foldability rapid test. *Feature is still in development.*
#@markdown ---
#@markdown
#@markdown This step can take up to ~3 mins (in case of CPU colab enviroment ~5 min).
#@markdown
# #@markdown With installation of ESMfold ~7 mins.

import os

if not os.path.isfile("ENV_READY"):
  !pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 torch_geometric
  !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
  !pip install biopython pydssp
  !git clone https://github.com/AndreiSokolovskii/hackaton_december_2025.git
  !cp hackaton_december_2025/*.py .
  os.system('touch ENV_READY')

from colab_upload_helper import process_upload, create_output_folder,_read_meta


# ESM_fold_for_test = False #@param {type:"boolean"}
# version = "1"
# model_name = "esmfold.model"

# if ESM_fold_test:
#   import os, time
#   if not os.path.isfile(model_name):
#     # download esmfold params
#     os.system("apt-get install aria2 -qq")
#     os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} &")

#     if not os.path.isfile("finished_install"):
#       # install libs
#       print("installing libs...")
#       os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif")
#       os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git")

#       print("installing openfold...")
#       # install openfold
#       os.system(f"pip install -q git+https://github.com/sokrypton/openfold.git")

#       print("installing esmfold...")
#       # install esmfold
#       os.system(f"pip install -q git+https://github.com/sokrypton/esm.git")
#       os.system("touch finished_install")

#     # wait for Params to finish downloading...
#     while not os.path.isfile(model_name):
#       time.sleep(5)
#     if os.path.isfile(f"{model_name}.aria2"):
#       print("downloading params...")
#     while os.path.isfile(f"{model_name}.aria2"):
#       time.sleep(5)


**Instructions**
---
---

The use of `input` can be done as follows:
- `input = '6X9Z'` - sequence design for the protein backbone downloaded from RCSB
- `input = ''` - allows to upload the structure from local storage in format  ***.pdb*** or ***.zip*** with a lot of PDBs.
---
- `use_last = 'True / False'` - the last time uploaded file will be used for prediction to skip uploading.
---
- `design_position = 'all'` - design all residues in structure.
- `design_position = '11 12 14:18'` - designing position lists specified in python style.
---
- `regime = 'One Shot Fast'` - fast design regime with generating all sequences at once per structure.
- `regime = 'One Shot Diverse'` - slower regime with generating more diverse set of sequences, ***probably***, should be tested.
- `regime = 'Iterative sampling'` - the slowest regime with iterative generating sequences **inspired by ProteinMPNN behaviour**.
---

In [ ]:
#@markdown ##Settings and run.
input = '' #@param {type:"string"}
use_last = True #@param {type:"boolean"}

design_position = 'all'#@param {type:"string"}
#@markdown - Position lists, e.g. 11 12 14:18. Default = 0 => design all residues

regime = "One Shot Fast" #@param ["One Shot Fast", "One Shot Diverse", "Iterative sampling"]

path_list = process_upload(is_same=use_last, pdb_code=input)

model_version = "v1" #@param ["v1", "v2"]
#@markdown - `model_version = v1` - trained in fully-blind regime
#@markdown - `model_version = v2` - trained with 10% masking regime. More preferable for partial redesign

num_seq_per_target = 7 # @param {"type":"raw"}
#@markdown - `num_seq_per_target = '3'` -  3 sequences will be generated per structure.

sampling_temp = "0.1" #@param ["0.01", "0.1", "0.15", "0.2", "0.5", "0.7", "1", "1.5", "2"]
#@markdown - Sampling temperature T=0.0 means taking argmax, T \>> 1.0 means sample randomly.
rm_aa = "" #@param {type:"string"}
#@markdown - `rm_aa='C'` - do not use [C]ysteines.
progress_bar = True #@param {type:"boolean"}


from run import main as run

class colab_args(object):
  def __init__(self, regime, model_version, num_seq_per_target,
               sampling_temp, rm_aa, progress_bar, design_position):
    short_regime_names = {"One Shot Fast":'OSF', "One Shot Diverse": 'OSD', "Iterative sampling":'IS'}

    self.model_masked = True if model_version == 'v1' else False
    self.num_seq_per_target = int(num_seq_per_target)
    self.temperature = float(sampling_temp)
    self.suppress_AAs = [aa for aa in rm_aa] if len(rm_aa) > 0 else ['X']
    self.verbose = progress_bar
    self.design_position = str(design_position) if design_position != 'all' else '0'
    self.iterative_sampling = True if regime == 'Iterative sampling' else False
    self.one_shot_diverse = True if regime == 'One Shot Diverse' else False
    meta = _read_meta()
    self.input_pdb = str(path_list)
    if meta['last'][-4:] == '.pdb':
      output_path = meta['last'][:-4]
    else:
      self.input_pdb = self.input_pdb + '/*.pdb'
      output_path = meta['last']

    output_path = output_path + '_r'\
     + short_regime_names[regime] + '_s'\
     + str(self.num_seq_per_target) + '_t' \
     + str(self.temperature) + '_m_' + str(model_version)
    self.output_path = str(create_output_folder(output_path))

    self.seed = 42
    self.path_to_model = 'hackaton_december_2025/weights'


args = colab_args(regime=regime, model_version=model_version,
                  num_seq_per_target=num_seq_per_target, sampling_temp=sampling_temp,
                  rm_aa=rm_aa, progress_bar=progress_bar, design_position=design_position,
                  )
run(args)

In [ ]:
#@title Download prediction

#@markdown Once this cell has been executed, predicted sequences from the last success run

#@markdown will be automatically downloaded  to your computer.
from google.colab import files
from colab_upload_helper import archive_latest_output
zip_file = archive_latest_output()
print("Last output zipped to:", zip_file)
files.download(zip_file)